In [1]:
import pandas as pd
import numpy as np

# ── 1. LOAD ──────────────────────────────────────────────────────────────────
try:
    df = pd.read_csv("data/gene properties/6_OmicsSomaticMutationsProfile.csv")
    if df.shape[1] == 1:
        raise ValueError
except:
    df = pd.read_csv("data/gene properties/6_OmicsSomaticMutationsProfile.csv", sep="\t")

print("=== SHAPE ===")
print(df.shape)

# ── 2. COLUMNS & DTYPES ──────────────────────────────────────────────────────
print("\n=== COLUMNS & DTYPES ===")
print(df.dtypes.to_string())

# ── 3. FIRST FEW ROWS ────────────────────────────────────────────────────────
print("\n=== HEAD (3) ===")
print(df.head(3).to_string())

# ── 4. NULLS ─────────────────────────────────────────────────────────────────
print("\n=== NULL COUNTS ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print(f"Total null cells: {df.isnull().sum().sum():,}")

# ── 5. UNIQUE VALUES PER COLUMN ──────────────────────────────────────────────
print("\n=== UNIQUE VALUE COUNTS PER COLUMN ===")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique")

# ── 6. CELL LINE ID FORMAT ───────────────────────────────────────────────────
print("\n=== CELL LINE ID FORMAT ===")
for col in df.columns[:6]:
    sample = df[col].dropna().astype(str).head(3).tolist()
    has_ach  = any(v.startswith('ACH-') for v in sample)
    has_cvcl = any('CVCL' in v for v in sample)
    print(f"  {col}: {sample}  ACH={has_ach} CVCL={has_cvcl}")



/var/folders/w4/fbz2zhr165g6wr5k09mtgl9r0000gn/T/ipykernel_27415/1702461793.py:6: DtypeWarning: Columns (22,50,54,56,57,58,59,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/gene properties/6_OmicsSomaticMutationsProfile.csv")


=== SHAPE ===
(1066869, 70)

=== COLUMNS & DTYPES ===
Chrom                               object
Pos                                  int64
Ref                                 object
Alt                                 object
AF                                 float64
DP                                   int64
RefCount                             int64
AltCount                             int64
GT                                  object
PS                                 float64
VariantType                         object
VariantInfo                         object
DNAChange                           object
ProteinChange                       object
HugoSymbol                          object
Exon                                object
Intron                              object
EnsemblGeneID                       object
EnsemblFeatureID                    object
HgncName                            object
HgncFamily                          object
UniprotID                           object


In [2]:
# ── 7. KEY MUTATION COLUMNS ──────────────────────────────────────────────────
print("\n=== MUTATION TYPE / EFFECT VALUES ===")
for col in df.columns:
    if any(kw in col.lower() for kw in
           ['type', 'effect', 'class', 'variant', 'consequence',
            'category', 'status', 'filter', 'confidence']):
        print(f"\n  {col}:")
        print(df[col].value_counts().head(10).to_string())

# ── 8. GENE ID FORMAT ────────────────────────────────────────────────────────
print("\n=== GENE ID FORMAT ===")
for col in df.columns:
    if any(kw in col.lower() for kw in ['gene', 'hugo', 'symbol', 'ensg']):
        sample = df[col].dropna().astype(str).head(5).tolist()
        print(f"  {col}: {sample}")

# ── 9. IsDefaultEntryForModel ────────────────────────────────────────────────
print("\n=== IsDefaultEntryForModel (if present) ===")
default_cols = [c for c in df.columns if 'default' in c.lower()]
if default_cols:
    for col in default_cols:
        print(f"  {col}:")
        print(df[col].value_counts().to_string())
else:
    print("  Column not found")

# ── 10. CELL LINE COVERAGE ───────────────────────────────────────────────────
print("\n=== CELL LINE COVERAGE ===")
for col in df.columns:
    sample = df[col].dropna().astype(str).head(20)
    if sample.str.startswith('ACH-').any():
        print(f"  Cell line column: '{col}'")
        print(f"  Unique cell lines: {df[col].nunique()}")
        print(f"  Sample: {df[col].dropna().unique()[:5].tolist()}")
        break

# ── 11. FORMAT GUESS ─────────────────────────────────────────────────────────
print("\n=== FORMAT GUESS ===")
print(f"Rows: {df.shape[0]:,}  Cols: {df.shape[1]}")
if df.shape[0] > df.shape[1]:
    print("→ Likely LONG format (one row per mutation event)")
else:
    print("→ Likely WIDE format")

# ── 12. GROUND TRUTH SPOT CHECK ──────────────────────────────────────────────
print("\n=== CANONICAL MUTATION SPOT CHECK ===")
df_str = df.astype(str)
for gene in ['TP53', 'KRAS', 'BRAF', 'EGFR', 'PIK3CA', 'PTEN']:
    found = df_str.apply(
        lambda col: col.str.contains(gene, na=False)).any().any()
    print(f"  {gene}: {'FOUND' if found else 'not found'}")


=== MUTATION TYPE / EFFECT VALUES ===

  VariantType:
VariantType
SNV             932636
deletion         68068
substitution     42371
insertion        23794

  VariantInfo:
VariantInfo
missense_variant                          857307
frameshift_variant                         74994
stop_gained                                55519
missense_variant&splice_region_variant     25557
splice_acceptor_variant                    14697
splice_donor_variant                       13330
inframe_deletion                            7028
start_lost                                  2151
inframe_insertion                           2127
stop_gained&splice_region_variant           2067

  MolecularConsequence:
MolecularConsequence
SO:0001583|missense_variant                                                                       22026
SO:0001589|frameshift_variant                                                                      2717
SO:0001583|missense_variant,SO:0001619|non-coding_transcript_variant 

In [5]:
import pandas as pd

df = pd.read_csv("data/gene properties/6_OmicsSomaticMutationsProfile.csv", low_memory=False)

# ── 1. RNA profile overlap ──────────────────────────────────────────────────
profiles_file8 = pd.read_csv("data/nomenclature/8_DepMap_OmicsProfiles.csv")
rna_profile_ids = set(profiles_file8[profiles_file8["Datatype"] == "rna"]["ProfileID"])

mut_profile_ids = set(df["ProfileID"].dropna())
overlap = mut_profile_ids & rna_profile_ids
only_mut = mut_profile_ids - rna_profile_ids
only_rna = rna_profile_ids - mut_profile_ids

print("=== RNA PROFILE OVERLAP ===")
print(f"  Unique ProfileIDs in mutations file:  {len(mut_profile_ids):,}")
print(f"  Unique RNA ProfileIDs (File 8):        {len(rna_profile_ids):,}")
print(f"  Overlap (both RNA + mutations):        {len(overlap):,}")
print(f"  Only in mutations (WES/WGS only):      {len(only_mut):,}")
print(f"  Only in RNA (no mutation data):        {len(only_rna):,}")

# ── 2. VepImpact distribution ───────────────────────────────────────────────
print("\n=== VepImpact DISTRIBUTION ===")
print(df["VepImpact"].value_counts(dropna=False).to_string())

# count of rows that would survive HIGH + MODERATE filter
high_mod = df["VepImpact"].isin(["HIGH", "MODERATE"])
print(f"\n  Rows kept at HIGH/MODERATE:  {high_mod.sum():,}  ({high_mod.mean()*100:.1f}%)")

# ── 3. ProteinChange format spot-check ──────────────────────────────────────
print("\n=== ProteinChange FORMAT SPOT-CHECK ===")

genes_of_interest = {
    "EGFR":   ["p.E746_A750del", "p.L858R"],
    "KRAS":   ["p.G12D", "p.G12V", "p.G13D"],
    "BRAF":   ["p.V600E"],
    "TP53":   ["p.R175H", "p.R248W"],
    "PIK3CA": ["p.E545K", "p.H1047R"],
}

for gene, expected in genes_of_interest.items():
    subset = df[df["HugoSymbol"] == gene]["ProteinChange"].dropna()
    top5 = subset.value_counts().head(5).index.tolist()
    hits = [p for p in expected if p in subset.values]
    print(f"\n  {gene}:")
    print(f"    Top-5 ProteinChange values: {top5}")
    print(f"    Expected hotspots found:    {hits if hits else 'NONE — check notation'}")

# ── 4. EnsemblGeneID quality check ──────────────────────────────────────────
print("\n=== EnsemblGeneID QUALITY ===")

ensg_col = df["EnsemblGeneID"]
null_count  = ensg_col.isna().sum()
versioned   = ensg_col.dropna().str.contains(r"\.", na=False).sum()   # e.g. ENSG00000187634.13
clean_ensg  = ensg_col.dropna().str.match(r"^ENSG\d+$").sum()
malformed   = ensg_col.dropna()[~ensg_col.dropna().str.match(r"^ENSG\d+(\.\d+)?$")]

print(f"  Null EnsemblGeneID:         {null_count:,}")
print(f"  Clean (no version suffix):  {clean_ensg:,}")
print(f"  Versioned (ENSG.X):         {versioned:,}")
print(f"  Malformed (neither):        {len(malformed):,}")
if len(malformed) > 0:
    print(f"  Malformed examples:         {malformed.unique()[:5].tolist()}")

# Also check LofGeneId versioning
lof_versioned = df["LofGeneId"].dropna().str.contains(r"\.", na=False).sum()
lof_total     = df["LofGeneId"].dropna().shape[0]
print(f"\n  LofGeneId rows with version suffix: {lof_versioned:,} / {lof_total:,}")

# ── 5. OncogeneHighImpact / TumorSuppressorHighImpact flags ─────────────────
print("\n=== ONCOGENE / TSG FLAG DISTRIBUTION ===")

onco = df["OncogeneHighImpact"].value_counts(dropna=False)
tsg  = df["TumorSuppressorHighImpact"].value_counts(dropna=False)
both = (df["OncogeneHighImpact"] & df["TumorSuppressorHighImpact"]).sum()

print(f"  OncogeneHighImpact:\n{onco.to_string()}")
print(f"\n  TumorSuppressorHighImpact:\n{tsg.to_string()}")
print(f"\n  Flagged TRUE for BOTH:  {both:,}  (sanity check — should be rare)")

# Top genes flagged as oncogene high impact
print("\n  Top 10 genes with OncogeneHighImpact=True:")
print(
    df[df["OncogeneHighImpact"]==True]["HugoSymbol"]
    .value_counts().head(10).to_string()
)

print("\n  Top 10 genes with TumorSuppressorHighImpact=True:")
print(
    df[df["TumorSuppressorHighImpact"]==True]["HugoSymbol"]
    .value_counts().head(10).to_string()
)

# ── 6. VepBiotype filter preview ────────────────────────────────────────────
print("\n=== VepBiotype FILTER PREVIEW ===")
biotype_counts = df["VepBiotype"].value_counts(dropna=False)
print(biotype_counts.head(15).to_string())

pc_mask = df["VepBiotype"].isin(["protein_coding", "protein_coding_LoF"])
print(f"\n  Rows kept at protein_coding / protein_coding_LoF:  {pc_mask.sum():,}  ({pc_mask.mean()*100:.1f}%)")

=== RNA PROFILE OVERLAP ===
  Unique ProfileIDs in mutations file:  2,828
  Unique RNA ProfileIDs (File 8):        1,495
  Overlap (both RNA + mutations):        0
  Only in mutations (WES/WGS only):      2,828
  Only in RNA (no mutation data):        1,495

=== VepImpact DISTRIBUTION ===
VepImpact
MODERATE    892565
HIGH        173751
MODIFIER       514
LOW             39

  Rows kept at HIGH/MODERATE:  1,066,316  (99.9%)

=== ProteinChange FORMAT SPOT-CHECK ===

  EGFR:
    Top-5 ProteinChange values: ['p.E746_A750del', 'p.A289V', 'p.V592F', 'p.L858R', 'p.C523R']
    Expected hotspots found:    ['p.E746_A750del', 'p.L858R']

  KRAS:
    Top-5 ProteinChange values: ['p.G12D', 'p.G12V', 'p.G12C', 'p.G13D', 'p.G12A']
    Expected hotspots found:    ['p.G12D', 'p.G12V', 'p.G13D']

  BRAF:
    Top-5 ProteinChange values: ['p.V600E', 'p.P403LfsTer8', 'p.N486_P490del', 'p.V600D', 'p.V600K']
    Expected hotspots found:    ['p.V600E']

  TP53:
    Top-5 ProteinChange values: ['p.R248Q', 'p.R

In [6]:
# Bridge: mutations ProfileID → ACH- → check against RNA cell lines
mut_to_ach = profiles_file8[["ProfileID", "ModelID"]].copy()

# Get ACH- IDs for mutation profiles (WES + WGS)
mut_ach_ids = set(
    mut_to_ach[mut_to_ach["ProfileID"].isin(mut_profile_ids)]["ModelID"].dropna()
)

# Get ACH- IDs for RNA profiles
rna_ach_ids = set(
    mut_to_ach[mut_to_ach["ProfileID"].isin(rna_profile_ids)]["ModelID"].dropna()
)

overlap_ach = mut_ach_ids & rna_ach_ids

print(f"Cell lines with BOTH mutation + RNA data: {len(overlap_ach):,}")
print(f"Cell lines with mutations only:           {len(mut_ach_ids - rna_ach_ids):,}")
print(f"Cell lines with RNA only:                 {len(rna_ach_ids - mut_ach_ids):,}")

Cell lines with BOTH mutation + RNA data: 1,407
Cell lines with mutations only:           337
Cell lines with RNA only:                 72
